In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import to_date, lit, date_format

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("BookCredit") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/17 19:22:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
credit = spark.read \
    .parquet("/data/processed/credit_card")

credit.createOrReplaceTempView("credit")

# Contagem de linhas e colunas
num_rows = credit.count()
num_columns = len(credit.columns)

print(f'Quantidade de linhas: {num_rows}')
print(f'Quantidade de variaveis (colunas): {num_columns}')

credit.show(5, truncate=False)

Quantidade de linhas: 3840312
Quantidade de variaveis (colunas): 24
+-------------+----------+----------------------+----------------------------------+-----------------------------------+-------------------------------+-------------------------------------+-----------------------------------+----------------------------------+------------------------------+------------------------------------+-----------------------------------+------------------------+-------------------------------+-----------------------------------+-------------------------------+-------------------------------------+-----------------------------------+------------------------------------+------------------------------+-----------------+---------------------+-------------------+-----------------+
|PK_AGG_CREDIT|SK_ID_CURR|FVL_AMT_BALANCE_CREDIT|FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT|FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT|FVL_AMT_DRAWINGS_CURRENT_CREDIT|FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT|FVL_AMT_DRAWINGS_POS_CURRENT_CR

In [3]:
qtd_linhas = credit.count()

In [4]:
spark.sql("""
            Select
                FC_NAME_CONTRACT_STATUS_CREDIT,
                count(*) as VOLUME,
                round(100*(count(*)/{}),2) as VOL_PERCENT
            from 
                credit
            group by 
                FC_NAME_CONTRACT_STATUS_CREDIT
            order by 
                VOLUME desc
""".format(qtd_linhas)).show(50,False)

+------------------------------+-------+-----------+
|FC_NAME_CONTRACT_STATUS_CREDIT|VOLUME |VOL_PERCENT|
+------------------------------+-------+-----------+
|Active                        |3698436|96.31      |
|Completed                     |128918 |3.36       |
|Signed                        |11058  |0.29       |
|Demand                        |1365   |0.04       |
|Sent proposal                 |513    |0.01       |
|Refused                       |17     |0.0        |
|Approved                      |5      |0.0        |
+------------------------------+-------+-----------+



In [6]:
credit.printSchema()

root
 |-- PK_AGG_CREDIT: integer (nullable = true)
 |-- SK_ID_CURR: integer (nullable = true)
 |-- FVL_AMT_BALANCE_CREDIT: double (nullable = true)
 |-- FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT: integer (nullable = true)
 |-- FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT: double (nullable = true)
 |-- FVL_AMT_DRAWINGS_CURRENT_CREDIT: double (nullable = true)
 |-- FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT: double (nullable = true)
 |-- FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT: double (nullable = true)
 |-- FVL_AMT_INST_MIN_REGULARITY_CREDIT: double (nullable = true)
 |-- FVL_AMT_PAYMENT_CURRENT_CREDIT: double (nullable = true)
 |-- FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT: double (nullable = true)
 |-- FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT: double (nullable = true)
 |-- FVL_AMT_RECIVABLE_CREDIT: double (nullable = true)
 |-- FVL_AMT_TOTAL_RECEIVABLE_CREDIT: double (nullable = true)
 |-- FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT: double (nullable = true)
 |-- FVL_CNT_DRAWINGS_CURRENT_CREDIT: integer (nullable = true)
 |-- F

In [7]:
credit_01 = spark.sql("""
    SELECT
        PK_AGG_CREDIT,

        -- Estatísticas gerais de valores dos créditos anteriores
        min(FVL_AMT_BALANCE_CREDIT) as MIN_FVL_AMT_BALANCE_CREDIT,
        max(FVL_AMT_BALANCE_CREDIT) as MAX_FVL_AMT_BALANCE_CREDIT,
        avg(FVL_AMT_BALANCE_CREDIT) as AVG_FVL_AMT_BALANCE_CREDIT,
        
        min(FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT) as MIN_FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT,
        max(FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT) as MAX_FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT,
        avg(FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT) as AVG_FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT,
        
        min(FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT) as MIN_FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT,
        max(FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT) as MAX_FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT,
        avg(FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT) as AVG_FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT,
        
        min(FVL_AMT_DRAWINGS_CURRENT_CREDIT) as MIN_FVL_AMT_DRAWINGS_CURRENT_CREDIT,
        max(FVL_AMT_DRAWINGS_CURRENT_CREDIT) as MAX_FVL_AMT_DRAWINGS_CURRENT_CREDIT,
        avg(FVL_AMT_DRAWINGS_CURRENT_CREDIT) as AVG_FVL_AMT_DRAWINGS_CURRENT_CREDIT,
        
        min(FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT) as MIN_FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT,
        max(FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT) as MAX_FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT,
        avg(FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT) as AVG_FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT,
        
        min(FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT) as MIN_FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT,
        max(FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT) as MAX_FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT,
        avg(FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT) as AVG_FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT,
        
        min(FVL_AMT_INST_MIN_REGULARITY_CREDIT) as MIN_FVL_AMT_INST_MIN_REGULARITY_CREDIT,
        max(FVL_AMT_INST_MIN_REGULARITY_CREDIT) as MAX_FVL_AMT_INST_MIN_REGULARITY_CREDIT,
        avg(FVL_AMT_INST_MIN_REGULARITY_CREDIT) as AVG_FVL_AMT_INST_MIN_REGULARITY_CREDIT,
        
        min(FVL_AMT_PAYMENT_CURRENT_CREDIT) as MIN_FVL_AMT_PAYMENT_CURRENT_CREDIT,
        max(FVL_AMT_PAYMENT_CURRENT_CREDIT) as MAX_FVL_AMT_PAYMENT_CURRENT_CREDIT,
        avg(FVL_AMT_PAYMENT_CURRENT_CREDIT) as AVG_FVL_AMT_PAYMENT_CURRENT_CREDIT,
        
        min(FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT) as MIN_FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT,
        max(FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT) as MAX_FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT,
        avg(FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT) as AVG_FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT,
        
        min(FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT) as MIN_FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT,
        max(FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT) as MAX_FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT,
        avg(FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT) as AVG_FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT,
        
        min(FVL_AMT_RECIVABLE_CREDIT) as MIN_FVL_AMT_RECIVABLE_CREDIT,
        max(FVL_AMT_RECIVABLE_CREDIT) as MAX_FVL_AMT_RECIVABLE_CREDIT,
        avg(FVL_AMT_RECIVABLE_CREDIT) as AVG_FVL_AMT_RECIVABLE_CREDIT,
        
        min(FVL_AMT_TOTAL_RECEIVABLE_CREDIT) as MIN_FVL_AMT_TOTAL_RECEIVABLE_CREDIT,
        max(FVL_AMT_TOTAL_RECEIVABLE_CREDIT) as MAX_FVL_AMT_TOTAL_RECEIVABLE_CREDIT,
        avg(FVL_AMT_TOTAL_RECEIVABLE_CREDIT) as AVG_FVL_AMT_TOTAL_RECEIVABLE_CREDIT,
        
        min(FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT) as MIN_FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT,
        max(FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT) as MAX_FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT,
        avg(FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT) as AVG_FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT,
        
        min(FVL_CNT_DRAWINGS_CURRENT_CREDIT) as MIN_FVL_CNT_DRAWINGS_CURRENT_CREDIT,
        max(FVL_CNT_DRAWINGS_CURRENT_CREDIT) as MAX_FVL_CNT_DRAWINGS_CURRENT_CREDIT,
        avg(FVL_CNT_DRAWINGS_CURRENT_CREDIT) as AVG_FVL_CNT_DRAWINGS_CURRENT_CREDIT,
        
        min(FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT) as MIN_FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT,
        max(FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT) as MAX_FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT,
        avg(FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT) as AVG_FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT,
        
        min(FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT) as MIN_FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT,
        max(FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT) as MAX_FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT,
        avg(FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT) as AVG_FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT,
        
        min(FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT) as MIN_FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT,
        max(FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT) as MAX_FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT,
        avg(FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT) as AVG_FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT,
        
        min(FVL_SK_DPD_CREDIT) as MIN_FVL_SK_DPD_CREDIT,
        max(FVL_SK_DPD_CREDIT) as MAX_FVL_SK_DPD_CREDIT,
        avg(FVL_SK_DPD_CREDIT) as AVG_FVL_SK_DPD_CREDIT,
        
        min(FVL_SK_DPD_DEF_CREDIT) as MIN_FVL_SK_DPD_DEF_CREDIT,
        max(FVL_SK_DPD_DEF_CREDIT) as MAX_FFVL_SK_DPD_DEF_CREDIT,
        avg(FVL_SK_DPD_DEF_CREDIT) as AVG_FVL_SK_DPD_DEF_CREDIT,



        --Quantidade de créditos por tipo
        sum(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' then 1 else 0 end) as QT_CONTRACT_STATUS_ACTIVE_POS,
        sum(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Completed' then 1 else 0 end) as QT_CONTRACT_STATUS_COMPLETED_POS,


        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_AMT_BALANCE_CREDIT else null end) as AVG_AMT_BALANCE_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_BALANCE_CREDIT else null end) as AVG_AMT_BALANCE_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_BALANCE_CREDIT else null end) as AVG_AMT_BALANCE_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_BALANCE_CREDIT else null end) as AVG_AMT_BALANCE_ACTIVE_U12M_CREDIT,
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT else null end) as AVG_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT else null end) as AVG_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT else null end) as AVG_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT else null end) as AVG_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_U12M_CREDIT,
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_U12M_CREDIT,
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_CURRENT_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_CURRENT_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_CURRENT_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_CURRENT_ACTIVE_U12M_CREDIT,
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_U12M_CREDIT,
                 
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_POS_CURRENT_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_POS_CURRENT_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_POS_CURRENT_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT else null end) as AVG_AMT_DRAWINGS_POS_CURRENT_ACTIVE_U12M_CREDIT,
                 
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_AMT_INST_MIN_REGULARITY_CREDIT else null end) as AVG_AMT_INST_MIN_REGULARITY_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_INST_MIN_REGULARITY_CREDIT else null end) as AVG_AMT_INST_MIN_REGULARITY_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_INST_MIN_REGULARITY_CREDIT else null end) as AVG_AMT_INST_MIN_REGULARITY_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_INST_MIN_REGULARITY_CREDIT else null end) as AVG_AMT_INST_MIN_REGULARITY_ACTIVE_U12M_CREDIT,
                 
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_AMT_PAYMENT_CURRENT_CREDIT else null end) as AVG_AMT_PAYMENT_CURRENT_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_PAYMENT_CURRENT_CREDIT else null end) as AVG_AMT_PAYMENT_CURRENT_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_PAYMENT_CURRENT_CREDIT else null end) as AVG_AMT_PAYMENT_CURRENT_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_PAYMENT_CURRENT_CREDIT else null end) as AVG_AMT_PAYMENT_CURRENT_ACTIVE_U12M_CREDIT,
                 
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT else null end) as AVG_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT else null end) as AVG_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT else null end) as AVG_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT else null end) as AVG_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_U12M_CREDIT,
                 
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT else null end) as AVG_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT else null end) as AVG_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT else null end) as AVG_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT else null end) as AVG_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_U12M_CREDIT,
                 
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_AMT_RECIVABLE_CREDIT else null end) as AVG_AMT_RECIVABLE_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_RECIVABLE_CREDIT else null end) as AVG_AMT_RECIVABLE_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_RECIVABLE_CREDIT else null end) as AVG_AMT_RECIVABLE_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_RECIVABLE_CREDIT else null end) as AVG_AMT_RECIVABLE_ACTIVE_U12M_CREDIT,
                 
                 
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_AMT_TOTAL_RECEIVABLE_CREDIT else null end) as AVG_AMT_TOTAL_RECEIVABLE_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_TOTAL_RECEIVABLE_CREDIT else null end) as AVG_AMT_TOTAL_RECEIVABLE_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_TOTAL_RECEIVABLE_CREDIT else null end) as AVG_AMT_TOTAL_RECEIVABLE_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_TOTAL_RECEIVABLE_CREDIT else null end) as AVG_AMT_TOTAL_RECEIVABLE_ACTIVE_U12M_CREDIT,              
                 
                 
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_U12M_CREDIT,
                 
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_CURRENT_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_CURRENT_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_CURRENT_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_CURRENT_ACTIVE_U12M_CREDIT,
                 
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_U12M_CREDIT,
                 
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_POS_CURRENT_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_POS_CURRENT_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_POS_CURRENT_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT else null end) as AVG_CNT_DRAWINGS_POS_CURRENT_ACTIVE_U12M_CREDIT,
                 
                 
        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT else null end) as AVG_CNT_INSTALMENT_MATURE_CUM_ACTIVE_U3M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT else null end) as AVG_CNT_INSTALMENT_MATURE_CUM_ACTIVE_U6M_CREDIT,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT else null end) as AVG_CNT_INSTALMENT_MATURE_CUM_ACTIVE_U9M_CREDIT, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when FC_NAME_CONTRACT_STATUS_CREDIT = 'Active' and PK_DATPART_CREDIT in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT else null end) as AVG_CNT_INSTALMENT_MATURE_CUM_ACTIVE_U12M_CREDIT                
                
    

    FROM credit
    GROUP BY PK_AGG_CREDIT
""")

credit_01.createOrReplaceTempView("credit_01")
num_rows = credit_01.count()
num_columns = len(credit_01.columns)

print(f'Quantidade de linhas: {num_rows}')
print(f'Quantidade de variaveis (colunas): {num_columns}')

25/07/17 20:07:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Quantidade de linhas: 104307
Quantidade de variaveis (colunas): 128


In [8]:
credit_01.show(5, truncate=False)

[Stage 26:>                                                         (0 + 1) / 1]

+-------------+--------------------------+--------------------------+--------------------------+--------------------------------------+--------------------------------------+--------------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+-----------------------------------+-----------------------------------+-----------------------------------+-----------------------------------------+-----------------------------------------+-----------------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+----------------------------------+----------------------------------+----------------------------------+----------------------------------------+----------------------------------------+---------

In [9]:
credit_01.printSchema()

root
 |-- PK_AGG_CREDIT: integer (nullable = true)
 |-- MIN_FVL_AMT_BALANCE_CREDIT: double (nullable = true)
 |-- MAX_FVL_AMT_BALANCE_CREDIT: double (nullable = true)
 |-- AVG_FVL_AMT_BALANCE_CREDIT: double (nullable = true)
 |-- MIN_FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT: integer (nullable = true)
 |-- MAX_FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT: integer (nullable = true)
 |-- AVG_FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT: double (nullable = true)
 |-- MIN_FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT: double (nullable = true)
 |-- MAX_FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT: double (nullable = true)
 |-- AVG_FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT: double (nullable = true)
 |-- MIN_FVL_AMT_DRAWINGS_CURRENT_CREDIT: double (nullable = true)
 |-- MAX_FVL_AMT_DRAWINGS_CURRENT_CREDIT: double (nullable = true)
 |-- AVG_FVL_AMT_DRAWINGS_CURRENT_CREDIT: double (nullable = true)
 |-- MIN_FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT: double (nullable = true)
 |-- MAX_FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT: double (nullable = true)
 |-

In [10]:
credit_02 = spark.sql("""
    SELECT
        *,
        -- Razões entre médias por janelas de tempo (tendência temporal)
        round(AVG_AMT_BALANCE_ACTIVE_U3M_CREDIT/AVG_AMT_BALANCE_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_AMT_BALANCE_ACTIVE_CREDIT,
        round(AVG_AMT_BALANCE_ACTIVE_U6M_CREDIT/AVG_AMT_BALANCE_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_AMT_BALANCE_ACTIVE_CREDIT,
        round(AVG_AMT_BALANCE_ACTIVE_U9M_CREDIT/AVG_AMT_BALANCE_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_AMT_BALANCE_ACTIVE_CREDIT,
        
        round(AVG_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_U3M_CREDIT/AVG_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_CREDIT,
        round(AVG_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_U6M_CREDIT/AVG_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_CREDIT,
        round(AVG_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_U9M_CREDIT/AVG_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_CREDIT,
        
        round(AVG_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_U3M_CREDIT/AVG_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_CREDIT,
        round(AVG_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_U6M_CREDIT/AVG_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_CREDIT,
        round(AVG_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_U9M_CREDIT/AVG_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_CREDIT,
        
        round(AVG_AMT_DRAWINGS_CURRENT_ACTIVE_U3M_CREDIT/AVG_AMT_DRAWINGS_CURRENT_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_AMT_DRAWINGS_CURRENT_ACTIVE_CREDIT,
        round(AVG_AMT_DRAWINGS_CURRENT_ACTIVE_U6M_CREDIT/AVG_AMT_DRAWINGS_CURRENT_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_AMT_DRAWINGS_CURRENT_ACTIVE_CREDIT,
        round(AVG_AMT_DRAWINGS_CURRENT_ACTIVE_U9M_CREDIT/AVG_AMT_DRAWINGS_CURRENT_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_AMT_DRAWINGS_CURRENT_ACTIVE_CREDIT,
        
        round(AVG_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_U3M_CREDIT/AVG_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_CREDIT,
        round(AVG_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_U6M_CREDIT/AVG_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_CREDIT,
        round(AVG_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_U9M_CREDIT/AVG_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_CREDIT,
        
        round(AVG_AMT_DRAWINGS_POS_CURRENT_ACTIVE_U3M_CREDIT/AVG_AMT_DRAWINGS_POS_CURRENT_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_AMT_DRAWINGS_POS_CURRENT_ACTIVE_CREDIT,
        round(AVG_AMT_DRAWINGS_POS_CURRENT_ACTIVE_U6M_CREDIT/AVG_AMT_DRAWINGS_POS_CURRENT_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_AMT_DRAWINGS_POS_CURRENT_ACTIVE_CREDIT,
        round(AVG_AMT_DRAWINGS_POS_CURRENT_ACTIVE_U9M_CREDIT/AVG_AMT_DRAWINGS_POS_CURRENT_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_AMT_DRAWINGS_POS_CURRENT_ACTIVE_CREDIT,
        
        round(AVG_AMT_INST_MIN_REGULARITY_ACTIVE_U3M_CREDIT/AVG_AMT_INST_MIN_REGULARITY_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_AMT_INST_MIN_REGULARITY_ACTIVE_CREDIT,
        round(AVG_AMT_INST_MIN_REGULARITY_ACTIVE_U6M_CREDIT/AVG_AMT_INST_MIN_REGULARITY_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_AMT_INST_MIN_REGULARITY_ACTIVE_CREDIT,
        round(AVG_AMT_INST_MIN_REGULARITY_ACTIVE_U9M_CREDIT/AVG_AMT_INST_MIN_REGULARITY_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_AMT_INST_MIN_REGULARITY_ACTIVE_CREDIT,
        
        round(AVG_AMT_PAYMENT_CURRENT_ACTIVE_U3M_CREDIT/AVG_AMT_PAYMENT_CURRENT_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_AMT_PAYMENT_CURRENT_ACTIVE_CREDIT,
        round(AVG_AMT_PAYMENT_CURRENT_ACTIVE_U6M_CREDIT/AVG_AMT_PAYMENT_CURRENT_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_AMT_PAYMENT_CURRENT_ACTIVE_CREDIT,
        round(AVG_AMT_PAYMENT_CURRENT_ACTIVE_U9M_CREDIT/AVG_AMT_PAYMENT_CURRENT_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_AMT_PAYMENT_CURRENT_ACTIVE_CREDIT,
        
        round(AVG_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_U3M_CREDIT/AVG_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_CREDIT,
        round(AVG_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_U6M_CREDIT/AVG_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_CREDIT,
        round(AVG_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_U9M_CREDIT/AVG_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_CREDIT,
        
        round(AVG_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_U3M_CREDIT/AVG_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_CREDIT,
        round(AVG_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_U6M_CREDIT/AVG_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_CREDIT,
        round(AVG_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_U9M_CREDIT/AVG_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_CREDIT,
        
        round(AVG_AMT_RECIVABLE_ACTIVE_U3M_CREDIT/AVG_AMT_RECIVABLE_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_AMT_RECIVABLE_ACTIVE_CREDIT,
        round(AVG_AMT_RECIVABLE_ACTIVE_U6M_CREDIT/AVG_AMT_RECIVABLE_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_AMT_RECIVABLE_ACTIVE_CREDIT,
        round(AVG_AMT_RECIVABLE_ACTIVE_U9M_CREDIT/AVG_AMT_RECIVABLE_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_AMT_RECIVABLE_ACTIVE_CREDIT,
        
        round(AVG_AMT_TOTAL_RECEIVABLE_ACTIVE_U3M_CREDIT/AVG_AMT_TOTAL_RECEIVABLE_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_AMT_TOTAL_RECEIVABLE_ACTIVE_CREDIT,
        round(AVG_AMT_TOTAL_RECEIVABLE_ACTIVE_U6M_CREDIT/AVG_AMT_TOTAL_RECEIVABLE_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_AMT_TOTAL_RECEIVABLE_ACTIVE_CREDIT,
        round(AVG_AMT_TOTAL_RECEIVABLE_ACTIVE_U9M_CREDIT/AVG_AMT_TOTAL_RECEIVABLE_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_AMT_TOTAL_RECEIVABLE_ACTIVE_CREDIT,
        
        round(AVG_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_U3M_CREDIT/AVG_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_CREDIT,
        round(AVG_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_U6M_CREDIT/AVG_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_CREDIT,
        round(AVG_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_U9M_CREDIT/AVG_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_CREDIT,
        
        round(AVG_CNT_DRAWINGS_CURRENT_ACTIVE_U3M_CREDIT/AVG_CNT_DRAWINGS_CURRENT_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_CNT_DRAWINGS_CURRENT_ACTIVE_CREDIT,
        round(AVG_CNT_DRAWINGS_CURRENT_ACTIVE_U6M_CREDIT/AVG_CNT_DRAWINGS_CURRENT_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_CNT_DRAWINGS_CURRENT_ACTIVE_CREDIT,
        round(AVG_CNT_DRAWINGS_CURRENT_ACTIVE_U9M_CREDIT/AVG_CNT_DRAWINGS_CURRENT_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_CNT_DRAWINGS_CURRENT_ACTIVE_CREDIT,
        
        round(AVG_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_U3M_CREDIT/AVG_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_CREDIT,
        round(AVG_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_U6M_CREDIT/AVG_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_CREDIT,
        round(AVG_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_U9M_CREDIT/AVG_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_CREDIT,
        
        round(AVG_CNT_DRAWINGS_POS_CURRENT_ACTIVE_U3M_CREDIT/AVG_CNT_DRAWINGS_POS_CURRENT_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_CNT_DRAWINGS_POS_CURRENT_ACTIVE_CREDIT,
        round(AVG_CNT_DRAWINGS_POS_CURRENT_ACTIVE_U6M_CREDIT/AVG_CNT_DRAWINGS_POS_CURRENT_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_CNT_DRAWINGS_POS_CURRENT_ACTIVE_CREDIT,
        round(AVG_CNT_DRAWINGS_POS_CURRENT_ACTIVE_U9M_CREDIT/AVG_CNT_DRAWINGS_POS_CURRENT_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_CNT_DRAWINGS_POS_CURRENT_ACTIVE_CREDIT,
        
        round(AVG_CNT_INSTALMENT_MATURE_CUM_ACTIVE_U3M_CREDIT/AVG_CNT_INSTALMENT_MATURE_CUM_ACTIVE_U6M_CREDIT,2) as VL_RAZ_MED_U3M_U6M_CNT_INSTALMENT_MATURE_CUM_ACTIVE_CREDIT,
        round(AVG_CNT_INSTALMENT_MATURE_CUM_ACTIVE_U6M_CREDIT/AVG_CNT_INSTALMENT_MATURE_CUM_ACTIVE_U9M_CREDIT,2) as VL_RAZ_MED_U6M_U9M_CNT_INSTALMENT_MATURE_CUM_ACTIVE_CREDIT,
        round(AVG_CNT_INSTALMENT_MATURE_CUM_ACTIVE_U9M_CREDIT/AVG_CNT_INSTALMENT_MATURE_CUM_ACTIVE_U12M_CREDIT,2) as VL_RAZ_MED_U9M_U12M_CNT_INSTALMENT_MATURE_CUM_ACTIVE_CREDIT

    FROM credit_01

""")

credit_02.createOrReplaceTempView("credit_02")
num_rows = credit_02.count()
num_columns = len(credit_02.columns)

print(f'Quantidade de linhas: {num_rows}')
print(f'Quantidade de variaveis (colunas): {num_columns}')

Quantidade de linhas: 104307
Quantidade de variaveis (colunas): 179


In [11]:
credit_02.show(5, truncate=False)

+-------------+--------------------------+--------------------------+--------------------------+--------------------------------------+--------------------------------------+--------------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+-----------------------------------+-----------------------------------+-----------------------------------+-----------------------------------------+-----------------------------------------+-----------------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+----------------------------------+----------------------------------+----------------------------------+----------------------------------------+----------------------------------------+---------


# 📘 Dicionário de Variáveis - Book Credit Card

| Variável | Descrição |
|----------|-----------|
| `MIN_FVL_AMT_BALANCE_CREDIT` | Menor saldo registrado durante o mês no crédito anterior |
| `MAX_FVL_AMT_BALANCE_CREDIT` | Maior saldo registrado durante o mês no crédito anterior |
| `AVG_FVL_AMT_BALANCE_CREDIT` | Saldo médio durante o mês no crédito anterior |
| `MIN_FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT` | Menor limite de crédito registrado durante o mês |
| `MAX_FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT` | Maior limite de crédito registrado durante o mês |
| `AVG_FVL_AMT_CREDIT_LIMIT_ACTUAL_CREDIT` | Limite médio de crédito durante o mês |
| `MIN_FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT` | Menor valor sacado em ATM durante o mês |
| `MAX_FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT` | Maior valor sacado em ATM durante o mês |
| `AVG_FVL_AMT_DRAWINGS_ATM_CURRENT_CREDIT` | Valor médio sacado em ATM durante o mês |
| `MIN_FVL_AMT_DRAWINGS_CURRENT_CREDIT` | Menor valor total sacado durante o mês |
| `MAX_FVL_AMT_DRAWINGS_CURRENT_CREDIT` | Maior valor total sacado durante o mês |
| `AVG_FVL_AMT_DRAWINGS_CURRENT_CREDIT` | Valor médio total sacado durante o mês |
| `MIN_FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT` | Menor valor de outros saques durante o mês |
| `MAX_FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT` | Maior valor de outros saques durante o mês |
| `AVG_FVL_AMT_DRAWINGS_OTHER_CURRENT_CREDIT` | Valor médio de outros saques durante o mês |
| `MIN_FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT` | Menor valor gasto em compras POS durante o mês |
| `MAX_FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT` | Maior valor gasto em compras POS durante o mês |
| `AVG_FVL_AMT_DRAWINGS_POS_CURRENT_CREDIT` | Valor médio gasto em compras POS durante o mês |
| `MIN_FVL_AMT_INST_MIN_REGULARITY_CREDIT` | Menor valor de parcela mínima registrado |
| `MAX_FVL_AMT_INST_MIN_REGULARITY_CREDIT` | Maior valor de parcela mínima registrado |
| `AVG_FVL_AMT_INST_MIN_REGULARITY_CREDIT` | Valor médio de parcela mínima |
| `MIN_FVL_AMT_PAYMENT_CURRENT_CREDIT` | Menor valor pago durante o mês |
| `MAX_FVL_AMT_PAYMENT_CURRENT_CREDIT` | Maior valor pago durante o mês |
| `AVG_FVL_AMT_PAYMENT_CURRENT_CREDIT` | Valor médio pago durante o mês |
| `MIN_FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT` | Menor valor total pago durante o mês |
| `MAX_FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT` | Maior valor total pago durante o mês |
| `AVG_FVL_AMT_PAYMENT_TOTAL_CURRENT_CREDIT` | Valor médio total pago durante o mês |
| `MIN_FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT` | Menor valor a receber do principal |
| `MAX_FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT` | Maior valor a receber do principal |
| `AVG_FVL_AMT_RECEIVABLE_PRINCIPAL_CREDIT` | Valor médio a receber do principal |
| `MIN_FVL_AMT_RECIVABLE_CREDIT` | Menor valor total a receber |
| `MAX_FVL_AMT_RECIVABLE_CREDIT` | Maior valor total a receber |
| `AVG_FVL_AMT_RECIVABLE_CREDIT` | Valor médio total a receber |
| `MIN_FVL_AMT_TOTAL_RECEIVABLE_CREDIT` | Menor valor total recebível |
| `MAX_FVL_AMT_TOTAL_RECEIVABLE_CREDIT` | Maior valor total recebível |
| `AVG_FVL_AMT_TOTAL_RECEIVABLE_CREDIT` | Valor médio total recebível |
| `MIN_FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT` | Menor número de saques em ATM |
| `MAX_FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT` | Maior número de saques em ATM |
| `AVG_FVL_CNT_DRAWINGS_ATM_CURRENT_CREDIT` | Número médio de saques em ATM |
| `MIN_FVL_CNT_DRAWINGS_CURRENT_CREDIT` | Menor número total de saques |
| `MAX_FVL_CNT_DRAWINGS_CURRENT_CREDIT` | Maior número total de saques |
| `AVG_FVL_CNT_DRAWINGS_CURRENT_CREDIT` | Número médio total de saques |
| `MIN_FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT` | Menor número de outros saques |
| `MAX_FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT` | Maior número de outros saques |
| `AVG_FVL_CNT_DRAWINGS_OTHER_CURRENT_CREDIT` | Número médio de outros saques |
| `MIN_FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT` | Menor número de compras POS |
| `MAX_FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT` | Maior número de compras POS |
| `AVG_FVL_CNT_DRAWINGS_POS_CURRENT_CREDIT` | Número médio de compras POS |
| `MIN_FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT` | Menor número de parcelas pagas |
| `MAX_FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT` | Maior número de parcelas pagas |
| `AVG_FVL_CNT_INSTALMENT_MATURE_CUM_CREDIT` | Número médio de parcelas pagas |
| `MIN_FVL_SK_DPD_CREDIT` | Menor número de dias em atraso (DPD) |
| `MAX_FVL_SK_DPD_CREDIT` | Maior número de dias em atraso (DPD) |
| `AVG_FVL_SK_DPD_CREDIT` | Média de dias em atraso (DPD) |
| `MIN_FVL_SK_DPD_DEF_CREDIT` | Menor número de dias em atraso com tolerância |
| `MAX_FVL_SK_DPD_DEF_CREDIT` | Maior número de dias em atraso com tolerância |
| `AVG_FVL_SK_DPD_DEF_CREDIT` | Média de dias em atraso com tolerância |
| `QT_CONTRACT_STATUS_ACTIVE_POS` | Quantidade de contratos com status "Active" |
| `QT_CONTRACT_STATUS_COMPLETED_POS` | Quantidade de contratos com status "Completed" |
| `AVG_AMT_BALANCE_ACTIVE_U3M_CREDIT` | Saldo médio de créditos ativos nos últimos 3 meses |
| `AVG_AMT_BALANCE_ACTIVE_U6M_CREDIT` | Saldo médio de créditos ativos nos últimos 6 meses |
| `AVG_AMT_BALANCE_ACTIVE_U9M_CREDIT` | Saldo médio de créditos ativos nos últimos 9 meses |
| `AVG_AMT_BALANCE_ACTIVE_U12M_CREDIT` | Saldo médio de créditos ativos nos últimos 12 meses |
| `VL_RAZ_MED_U3M_U6M_AMT_BALANCE_ACTIVE_CREDIT` | Razão entre médias de saldo (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_AMT_BALANCE_ACTIVE_CREDIT` | Razão entre médias de saldo (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_AMT_BALANCE_ACTIVE_CREDIT` | Razão entre médias de saldo (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_CREDIT` | Razão entre médias de limite de crédito (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_CREDIT` | Razão entre médias de limite de crédito (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_AMT_CREDIT_LIMIT_ACTUAL_ACTIVE_CREDIT` | Razão entre médias de limite de crédito (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_CREDIT` | Razão entre médias de saques em ATM (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_CREDIT` | Razão entre médias de saques em ATM (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_AMT_DRAWINGS_ATM_CURRENT_ACTIVE_CREDIT` | Razão entre médias de saques em ATM (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_AMT_DRAWINGS_CURRENT_ACTIVE_CREDIT` | Razão entre médias de saques totais (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_AMT_DRAWINGS_CURRENT_ACTIVE_CREDIT` | Razão entre médias de saques totais (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_AMT_DRAWINGS_CURRENT_ACTIVE_CREDIT` | Razão entre médias de saques totais (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_CREDIT` | Razão entre médias de outros saques (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_CREDIT` | Razão entre médias de outros saques (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_AMT_DRAWINGS_OTHER_CURRENT_ACTIVE_CREDIT` | Razão entre médias de outros saques (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_AMT_DRAWINGS_POS_CURRENT_ACTIVE_CREDIT` | Razão entre médias de compras POS (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_AMT_DRAWINGS_POS_CURRENT_ACTIVE_CREDIT` | Razão entre médias de compras POS (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_AMT_DRAWINGS_POS_CURRENT_ACTIVE_CREDIT` | Razão entre médias de compras POS (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_AMT_INST_MIN_REGULARITY_ACTIVE_CREDIT` | Razão entre médias de parcelas mínimas (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_AMT_INST_MIN_REGULARITY_ACTIVE_CREDIT` | Razão entre médias de parcelas mínimas (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_AMT_INST_MIN_REGULARITY_ACTIVE_CREDIT` | Razão entre médias de parcelas mínimas (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_AMT_PAYMENT_CURRENT_ACTIVE_CREDIT` | Razão entre médias de pagamentos (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_AMT_PAYMENT_CURRENT_ACTIVE_CREDIT` | Razão entre médias de pagamentos (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_AMT_PAYMENT_CURRENT_ACTIVE_CREDIT` | Razão entre médias de pagamentos (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_CREDIT` | Razão entre médias de pagamentos totais (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_CREDIT` | Razão entre médias de pagamentos totais (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_AMT_PAYMENT_TOTAL_CURRENT_ACTIVE_CREDIT` | Razão entre médias de pagamentos totais (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_CREDIT` | Razão entre médias de principal recebível (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_CREDIT` | Razão entre médias de principal recebível (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_AMT_RECEIVABLE_PRINCIPAL_ACTIVE_CREDIT` | Razão entre médias de principal recebível (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_AMT_RECIVABLE_ACTIVE_CREDIT` | Razão entre médias de valores recebíveis (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_AMT_RECIVABLE_ACTIVE_CREDIT` | Razão entre médias de valores recebíveis (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_AMT_RECIVABLE_ACTIVE_CREDIT` | Razão entre médias de valores recebíveis (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_AMT_TOTAL_RECEIVABLE_ACTIVE_CREDIT` | Razão entre médias de totais recebíveis (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_AMT_TOTAL_RECEIVABLE_ACTIVE_CREDIT` | Razão entre médias de totais recebíveis (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_AMT_TOTAL_RECEIVABLE_ACTIVE_CREDIT` | Razão entre médias de totais recebíveis (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_CREDIT` | Razão entre médias de contagem de saques ATM (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_CREDIT` | Razão entre médias de contagem de saques ATM (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_CNT_DRAWINGS_ATM_CURRENT_ACTIVE_CREDIT` | Razão entre médias de contagem de saques ATM (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_CNT_DRAWINGS_CURRENT_ACTIVE_CREDIT` | Razão entre médias de contagem total de saques (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_CNT_DRAWINGS_CURRENT_ACTIVE_CREDIT` | Razão entre médias de contagem total de saques (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_CNT_DRAWINGS_CURRENT_ACTIVE_CREDIT` | Razão entre médias de contagem total de saques (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_CREDIT` | Razão entre médias de contagem de outros saques (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_CREDIT` | Razão entre médias de contagem de outros saques (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_CNT_DRAWINGS_OTHER_CURRENT_ACTIVE_CREDIT` | Razão entre médias de contagem de outros saques (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_CNT_DRAWINGS_POS_CURRENT_ACTIVE_CREDIT` | Razão entre médias de contagem de compras POS (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_CNT_DRAWINGS_POS_CURRENT_ACTIVE_CREDIT` | Razão entre médias de contagem de compras POS (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_CNT_DRAWINGS_POS_CURRENT_ACTIVE_CREDIT` | Razão entre médias de contagem de compras POS (9m/12m) |
| `VL_RAZ_MED_U3M_U6M_CNT_INSTALMENT_MATURE_CUM_ACTIVE_CREDIT` | Razão entre médias de parcelas pagas (3m/6m) |
| `VL_RAZ_MED_U6M_U9M_CNT_INSTALMENT_MATURE_CUM_ACTIVE_CREDIT` | Razão entre médias de parcelas pagas (6m/9m) |
| `VL_RAZ_MED_U9M_U12M_CNT_INSTALMENT_MATURE_CUM_ACTIVE_CREDIT` | Razão entre médias de parcelas pagas (9m/12m) |



#### Salvando tabela particionada (Parquet)

In [12]:
nm_path = '/data/books/credit_card'
credit_02.write.parquet(nm_path, mode='overwrite')
#bureau_etl_01.coalesce(1).write.parquet(nm_path, mode="overwrite")

In [13]:
spark.stop()